# Weather Trend Forecasting - Data Cleaning and Preprocessing

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Load the dataset from the data folder.
# This step is needed so the notebook can work with the weather observations.
project_root = Path.cwd()
data_path = project_root / "data" / "GlobalWeatherRepository.csv"
if not data_path.exists():
    data_path = project_root / ".." / "data" / "GlobalWeatherRepository.csv"

raw_df = pd.read_csv(data_path)

# Keep a copy of the original data for reference.
# This protects the source data from accidental changes during cleaning.
df = raw_df.copy()

# Inspect the dataset shape, column names, and data types.
# This step helps us understand the structure before preprocessing.
print("Rows before cleaning:", df.shape[0])
print("Columns before cleaning:", df.shape[1])
print("\nFirst rows:")
print(df.head(3))
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

ModuleNotFoundError: No module named 'numpy'

In [4]:
# Convert the timestamp column into datetime format.
# This is required for extracting time-based features later.
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

# Check missing values after the datetime conversion.
# This helps us measure how much repair is needed.
print("Missing values after datetime conversion:")
print(df.isnull().sum())

NameError: name 'pd' is not defined

In [5]:
# Replace invalid numeric placeholders such as -9999 with missing values.
# This ensures that impossible values do not distort the statistics.
invalid_value_columns = [
    "temperature_celsius",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "humidity",
    "cloud",
    "visibility_km",
    "uv_index",
    "air_quality_Carbon_Monoxide",
    "air_quality_Sulphur_dioxide",
    "air_quality_PM10",
]
for col in invalid_value_columns:
    if col in df.columns:
        df[col] = df[col].replace(-9999, np.nan)

print("Missing values after invalid value replacement:")
print(df[invalid_value_columns].isnull().sum())

NameError: name 'df' is not defined

In [ ]:
# Detect outliers using the interquartile range method.
# This helps remove extreme values that can harm forecasting quality.
outlier_features = ["temperature_celsius", "pressure_mb", "wind_kph", "precip_mm"]
outlier_summary = {}

for col in outlier_features:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_summary[col] = int(mask.sum())
        df.loc[mask, col] = np.nan

print("Outliers replaced with NaN:")
print(outlier_summary)
print("\nMissing values after outlier treatment:")
print(df[outlier_features].isnull().sum())

In [ ]:
# Fill remaining missing values with the median for numeric columns.
# This keeps the dataset complete without dropping rows.
numeric_columns = [
    "latitude",
    "longitude",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
    "temperature_celsius",
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Create time-based features from the timestamp.
# These help the model capture seasonal and daily patterns.
df["year"] = df["last_updated"].dt.year
df["month"] = df["last_updated"].dt.month
df["day"] = df["last_updated"].dt.day
df["hour"] = df["last_updated"].dt.hour

print("Missing values after imputation:")
print(df[numeric_columns + ["year", "month", "day", "hour"]].isnull().sum())
print("\nTime feature preview:")
print(df[["last_updated", "year", "month", "day", "hour"]].head())

In [ ]:
# Select the variables that are most useful for forecasting.
# These columns represent the main physical and temporal signals for the model.
model_features = [
    "latitude",
    "longitude",
    "month",
    "day",
    "hour",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
]
target_col = "temperature_celsius"

processed_df = df[model_features + [target_col]].copy()

print("Processed dataframe shape:", processed_df.shape)
print("\nProcessed dataframe preview:")
print(processed_df.head())

In [ ]:
# Standardize the selected numerical features.
# This makes the model less sensitive to the scale of different variables.
scaler = StandardScaler()
processed_df[model_features] = scaler.fit_transform(processed_df[model_features])

print("Scaled dataframe preview:")
print(processed_df.head())
print("\nFeature means after scaling:")
print(processed_df[model_features].mean().round(4))
print("\nFeature standard deviations after scaling:")
print(processed_df[model_features].std().round(4))

In [ ]:
## EDA - Temperature Time Series Analysis


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    df["last_updated"],
    df["temperature_celsius"]
)

plt.title("Temperature Over Time")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.xticks(rotation=45)
plt.grid()

plt.show()


In [ ]:
## precipitation change over time
plt.figure(figsize=(12, 5))
plt.plot(
    df["last_updated"],
    df["precip_mm"]
)
plt.title("Precipitation Over Time")
plt.xlabel("Date")
plt.ylabel("Precipitation (mm)")

plt.xticks(rotation=45)
plt.grid()

plt.show()

In [ ]:
# Select important numerical weather features
corr_columns = [
    "temperature_celsius",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index"
]
corr_matrix = df[corr_columns].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Weather Features Correlation Heatmap")
plt.show()
